# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [2]:
# --- Install (if needed) ---
!pip install huggingface_hub pandas pyarrow -q

# --- Auth via Colab Secrets ---
from google.colab import userdata
import os
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

import pandas as pd
import numpy as np

# --- Load March 2026 partition + dimension tables ---
df_march = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
)
dim_clients = pd.read_parquet("hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet")
dim_content = pd.read_parquet("hf://datasets/FlyRank/internship-warehouse/dim_content.parquet")

print(df_march.shape, dim_clients.shape, dim_content.shape)

# --- Rebuild the working df: join content_type, word_count, created_date; apply filters ---
df = df_march.copy()
df = df.merge(
    dim_content[['content_hash_id', 'content_type', 'word_count', 'content_created_date']],
    on='content_hash_id', how='left'
)
df = df[df['gsc_data_available'] == True]
df = df[df['gsc_impressions'] >= 500]

df['content_age_days'] = (
    pd.to_datetime(df['report_date']) - pd.to_datetime(df['content_created_date'])
).dt.days

df['ctr_check'] = df['gsc_clicks'] / df['gsc_impressions']

print(df.shape)

(9841378, 31) (104, 9) (519606, 26)
(101451, 36)


## Section 1: Distributions

**content_age_days** — Roughly even spread from 0–482 days, median 158. No sharp heavy tail:
the 95th percentile (390) sits reasonably close to the max (482).

**gsc_avg_position** — Heavy right tail. Median position is 5.2 (decent ranking), but the top
1% of rows rank as poorly as position 48–149. Most content ranks well; a minority ranks very badly.

**gsc_impressions** — Very heavy right tail. Median is 772 impressions, but the max (40,084) is
roughly 35x the 99th percentile (4,378). A small number of high-exposure outliers pull the mean
(1,045) well above the median.

**ctr_check** — Heavy tail with a floor effect: 25% of rows (all already filtered to 500+
impressions) have exactly 0 clicks. The remaining distribution stretches up to 8.7% CTR, far
above the 99th percentile of 2%.

**Practical note:** given the heavy tails on gsc_impressions and gsc_avg_position, any signal
test using these fields should watch for outliers dominating the mean — bucket-based tests
(quartiles/terciles) are safer than raw averages here.

In [3]:
import pandas as pd

# Recompute ctr on the full working df (not banned here — this is EDA, not a feature)
df['ctr_check'] = df['gsc_clicks'] / df['gsc_impressions']

fields_to_check = ['content_age_days', 'gsc_avg_position', 'gsc_impressions', 'ctr_check']

for col in fields_to_check:
    print(f"\n--- {col} ---")
    print(df[col].describe())
    print(f"95th pct: {df[col].quantile(0.95):.2f} | 99th pct: {df[col].quantile(0.99):.2f} | max: {df[col].max():.2f}")



--- content_age_days ---
count    101451.000000
mean        172.247045
std         110.896058
min           0.000000
25%          71.000000
50%         158.000000
75%         237.000000
max         482.000000
Name: content_age_days, dtype: float64
95th pct: 390.00 | 99th pct: 425.00 | max: 482.00

--- gsc_avg_position ---
count    101451.000000
mean         11.689158
std          12.841297
min           0.000000
25%           3.398684
50%           5.189953
75%          17.734620
max         149.058929
Name: gsc_avg_position, dtype: float64
95th pct: 39.20 | 99th pct: 48.43 | max: 149.06

--- gsc_impressions ---
count    101451.000000
mean       1044.690235
std        1011.497665
min         500.000000
25%         602.000000
50%         772.000000
75%        1141.000000
max       40084.000000
Name: gsc_impressions, dtype: float64
95th pct: 2364.00 | 99th pct: 4378.50 | max: 40084.00

--- ctr_check ---
count    101451.000000
mean          0.002804
std           0.003728
min           0

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [4]:
# --- Signal Test 1: Staleness — does older content have lower CTR? (refresh-flag logic) ---
df['age_bucket'] = pd.qcut(df['content_age_days'], q=4, labels=['newest25%', 'q2', 'q3', 'oldest25%'])

test1 = df.groupby('age_bucket', observed=True)['ctr_check'].agg(['mean', 'median', 'count'])
print("=== Test 1: Staleness vs CTR ===")
print(test1)
print(f"n = {len(df)}")


=== Test 1: Staleness vs CTR ===
                mean    median  count
age_bucket                           
newest25%   0.002993  0.001707  25424
q2          0.002318  0.001408  25575
q3          0.002643  0.001534  25366
oldest25%   0.003269  0.002087  25086
n = 101451


### Signal Test 1: Staleness vs CTR (flag-linked: refresh flags)

**Rule assumption being tested:** older content has lower CTR (the premise behind refresh flags).

**Bucket table (quartiles of content_age_days, n=101,451):**
- newest25%: mean CTR 0.00299, median 0.00171, n=25,424
- q2: mean CTR 0.00232, median 0.00141, n=25,575
- q3: mean CTR 0.00264, median 0.00153, n=25,366
- oldest25%: mean CTR 0.00327, median 0.00209, n=25,086

**Verdict: OPPOSITE**

The oldest quartile has the *highest* mean and median CTR, not the lowest. There's no clean
monotonic decline with age — CTR dips in the middle quartiles and recovers by the oldest bucket.
This contradicts the simple "content decays over time" assumption behind refresh flags. A
plausible explanation: older content that's still active/ranking may be older *because* it
performs well (survivorship), while newer content hasn't yet accumulated ranking signal. This
doesn't mean refresh flags are worthless — it means raw age alone isn't the right trigger; the
refresh logic likely needs a trend signal (declining CTR *for that specific piece* over time),
not absolute age.

In [5]:
# --- Signal Test 2: CTR-vs-position — does better ranking position mean higher CTR? (CTR-fix flag logic) ---
df['position_bucket'] = pd.qcut(df['gsc_avg_position'], q=4, labels=['best25%', 'q2', 'q3', 'worst25%'])

test2 = df.groupby('position_bucket', observed=True)['ctr_check'].agg(['mean', 'median', 'count'])
print("=== Test 2: Position vs CTR ===")
print(test2)
print(f"n = {len(df)}")

=== Test 2: Position vs CTR ===
                     mean    median  count
position_bucket                           
best25%          0.003904  0.002688  25363
q2               0.003200  0.002151  25363
q3               0.002838  0.001674  25362
worst25%         0.001273  0.000000  25363
n = 101451


### Signal Test 2: Position vs CTR (flag-linked: CTR-fix logic)

**Rule assumption being tested:** better search ranking position drives higher CTR (the premise
behind CTR-fix flags).

**Bucket table (quartiles of gsc_avg_position, n=101,451):**
- best25%: mean CTR 0.00390, median 0.00269, n=25,363
- q2: mean CTR 0.00320, median 0.00215, n=25,363
- q3: mean CTR 0.00284, median 0.00167, n=25,362
- worst25%: mean CTR 0.00127, median 0.00000, n=25,363

**Verdict: CONFIRMED**

Clean monotonic decline across all four buckets — no reversals, no dips. CTR in the best-position
quartile is roughly 3x higher than the worst-position quartile, and the worst quartile's median
CTR is literally zero (over half those rows get no clicks at all). This directly supports the
assumption behind CTR-fix flags: position is a real, strong, directional driver of CTR. Unlike
Test 1, this is a signal safe to build a rule on.

In [6]:
# --- Signal Test 3: Volume — does higher impression volume mean lower CTR? (quick-win flag logic) ---
df['volume_bucket'] = pd.qcut(df['gsc_impressions'], q=4, labels=['lowest25%', 'q2', 'q3', 'highest25%'])

test3 = df.groupby('volume_bucket', observed=True)['ctr_check'].agg(['mean', 'median', 'count'])
print("=== Test 3: Volume vs CTR ===")
print(test3)
print(f"n = {len(df)}")

=== Test 3: Volume vs CTR ===
                   mean    median  count
volume_bucket                           
lowest25%      0.002861  0.001802  25436
q2             0.002871  0.001546  25337
q3             0.002781  0.001770  25340
highest25%     0.002702  0.001631  25338
n = 101451


### Signal Test 3: Volume vs CTR (flag-linked: quick-win logic)

**Rule assumption being tested:** higher-impression content shows a distinct CTR pattern
(e.g., high volume "coasting" on exposure with weaker relative engagement) — the premise
quick-win flags often lean on.

**Bucket table (quartiles of gsc_impressions, n=101,451):**
- lowest25%: mean CTR 0.00286, median 0.00180, n=25,436
- q2: mean CTR 0.00287, median 0.00155, n=25,337
- q3: mean CTR 0.00278, median 0.00177, n=25,340
- highest25%: mean CTR 0.00270, median 0.00163, n=25,338

**Verdict: MIXED**

CTR is essentially flat across impression-volume quartiles — no clear rise or decline, values
stay within a narrow band (0.0027–0.0029) with no consistent direction. This means impression
volume alone doesn't predict CTR performance one way or the other. It doesn't invalidate
quick-win logic, but it does mean volume by itself isn't a strong standalone signal — the
"opportunity" signal likely comes from the *combination* of high volume with low CTR relative
to position (as tested in Test 2), not from volume in isolation. This is consistent with how
the original lane was framed: high impressions AND low engagement together, not either alone.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

## Section 3: The Flag-Linked Test

Two of my three signal tests were directly tied to real FlyRank flags:

- **Test 1 (staleness → refresh flags)** — OPPOSITE result. The data did not support the
  refresh flag's core assumption that older content has lower CTR.
- **Test 2 (position → CTR-fix flags)** — CONFIRMED result. The data strongly supported the
  CTR-fix flag's core assumption that worse ranking position drives lower CTR.

**Primary flag-linked test: Test 2 (position → CTR-fix logic).**

The CTR-fix flag assumes: if content ranks poorly, its CTR will suffer, and fixing the ranking
(or the content driving that ranking) should be the lever pulled. The bucket data confirms this
cleanly — a monotonic ~3x drop in mean CTR from best-position to worst-position quartiles, with
the worst quartile's median CTR sitting at exactly zero. This is not a borderline or noisy result;
it's one of the strongest, most consistent patterns in the whole audit.

**Does the data support the rule's assumption?** Yes, unambiguously, for Test 2. The CTR-fix
flag's underlying logic holds up under a direct bucket test with real n counts (~25,363 per
bucket) and no reversals across quartiles.

**Caveat worth carrying forward:** Test 1 shows that not every flag's assumption holds this
cleanly — staleness alone was a weaker, even backwards, signal. This is a useful contrast: it
means flag logic should be evaluated individually, not assumed to generalize just because one
flag family (CTR-fix) tested well.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

## Section 4: What This Means in Practice

For a content team: ranking position is the strongest, most reliable driver of CTR we found —
if you can only fix one thing, fixing rank (via the existing CTR-fix flag logic) is a safe bet
to move engagement. Content age alone is NOT a reliable signal for prioritizing refreshes — the
data actually leaned slightly the other way, so refresh decisions should look at trend (is this
piece's performance declining over time) rather than raw age. Impression volume alone doesn't
tell you much about CTR performance either way — it only becomes meaningful when paired with a
low-CTR-relative-to-position signal, which is the real "opportunity" pattern worth chasing.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.